<a href="https://colab.research.google.com/github/adriennie/vllm-lora-extension/blob/main/qwen_vllm_multi_adapter_serving.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Install vLLM (ensures it is present in the environment)
!pip install -q vllm

# Verify and print the installed vLLM version
import vllm
print(f"[CHECKPOINT 0] Active vLLM version installed: {vllm.__version__}")

[CHECKPOINT 0] Active vLLM version installed: 0.31.0


In [8]:
from google.colab import drive
import os

# Mount Google Drive correctly to /content/drive
drive.mount('/content/drive')

# Define the base path where your adapter folders are stored
ADAPTER_BASE_PATH = "/content/drive/MyDrive/results-2"

# Verify and map individual adapter paths
adapters = {
    "r4": os.path.join(ADAPTER_BASE_PATH, "lora_adapter_r4"),
    "r8": os.path.join(ADAPTER_BASE_PATH, "lora_adapter_r8"),
    "r16": os.path.join(ADAPTER_BASE_PATH, "lora_adapter_r16")
}

for name, path in adapters.items():
    print(f"Adapter {name} path exists: {os.path.exists(path)} ({path})")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Adapter r4 path exists: True (/content/drive/MyDrive/results-2/lora_adapter_r4)
Adapter r8 path exists: True (/content/drive/MyDrive/results-2/lora_adapter_r8)
Adapter r16 path exists: True (/content/drive/MyDrive/results-2/lora_adapter_r16)


In [10]:
import sys
import os

# --- Robust Colab / IPython Stream Patch for vLLM v1 Engine ---
try:
    import ipykernel.iostream
    # Force IPython OutStreams to report a valid file descriptor
    ipykernel.iostream.OutStream.fileno = lambda self: 1
except ImportError:
    pass

if not hasattr(sys.stdout, 'fileno'):
    sys.stdout.fileno = lambda: 1

if not hasattr(sys.stderr, 'fileno'):
    sys.stderr.fileno = lambda: 2
# -------------------------------------------------------------

from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print("[CHECKPOINT 3] Initializing vLLM engine with Multi-LoRA capability...")
llm = LLM(
    model=MODEL_ID,
    enable_lora=True,
    max_lora_rank=16,
    max_model_len=512,
    gpu_memory_utilization=0.85
)
print("[CHECKPOINT 3.2] vLLM engine initialized successfully.")

[CHECKPOINT 3] Initializing vLLM engine with Multi-LoRA capability...
(MainProcess pid=6871) INFO 10-10 14:34:13 [api_utils.py:286] non-default args: {'max_model_len': 512, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enable_lora': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
(MainProcess pid=6871) INFO 10-10 14:34:13 [model.py:699] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=6871) WARNING 10-10 14:34:13 [model.py:2389] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
(MainProcess pid=6871) WARNING 10-10 14:34:13 [model.py:2442] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=6871) INFO 10-10 14:34:13 [model.py:2109] Using max model len 512
(MainProcess pid=6871) INFO 10-10 14:34:13 [kernel.py:408] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
(E

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore pid=22330) INFO 10-10 14:35:06 [default_loader.py:484] Loading weights took 12.53 seconds
(EngineCore pid=22330) INFO 10-10 14:35:06 [punica_selector.py:20] Using PunicaWrapperGPU.
(EngineCore pid=22330) INFO 10-10 14:35:07 [model_runner.py:407] Model loading took 3.01 GiB memory and 46.458257 seconds
(EngineCore pid=22330) WARNING 10-10 14:35:07 [topk_topp_sampler.py:131] FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5; falling back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
(EngineCore pid=22330) INFO 10-10 14:35:07 [utils.py:343] Using LBNHC KV cache layout.
(EngineCore pid=22330) INFO 10-10 14:35:30 [backends.py:1090] Using cache directory: /root/.cache/vllm/torch_compile_cache/c068ea8581/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=22330) INFO 10-10 14:35:30 [backends.py:1151] Dynamo bytecode transform time: 18.51 s


[rank0]:W1010 14:35:38.652000 22330 torch/_inductor/utils.py:1953] Not enough SMs to use max_autotune_gemm mode


(EngineCore pid=22330) INFO 10-10 14:35:44 [backends.py:391] Compiling a graph for compile range (1, 8192) takes 13.29 s
(EngineCore pid=22330) INFO 10-10 14:35:51 [backends.py:916] collected artifacts: 29 entries, 3 artifacts, 4773306 bytes total
(EngineCore pid=22330) INFO 10-10 14:35:51 [decorators.py:717] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/5f2273eac189a4e426bc2b9e4b41f654d5719024abcefe068671ad4c2e73ba4b/rank_0_0/model
(EngineCore pid=22330) INFO 10-10 14:35:51 [monitor.py:53] torch.compile took 39.79 s in total
(EngineCore pid=22330) WARNING 10-10 14:35:51 [utils.py:276] Using default LoRA kernel configs
(EngineCore pid=22330) INFO 10-10 14:36:02 [monitor.py:81] Initial profiling/warmup run took 11.03 s


Capturing CUDA graphs (FULL): 100%|██████████| 2/2 [00:00<00:00,  6.51it/s]


(EngineCore pid=22330) INFO 10-10 14:37:13 [model_runner.py:1087] Graph capturing finished in 63 secs, took 0.62 GiB
(EngineCore pid=22330) INFO 10-10 14:37:14 [gpu_worker.py:692] Available KV cache memory: 7.78 GiB
(EngineCore pid=22330) INFO 10-10 14:37:14 [gpu_worker.py:707] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.8500 is equivalent to --gpu-memory-utilization=0.8011 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.8989. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=22330) INFO 10-10 14:37:14 [kv_cache_utils.py:2464] GPU KV cache size: 291,504 tokens, Maximum concurrency for 512 tokens per request: 569.34x
(EngineCore pid=22330) INFO 10-10 14:37:14 [kernel_warmup.py:172] JIT kernel warmup starting.
(EngineCore pid=22330) INFO 10-10 14:37:14 [kernel_warmup.py:185] JIT kernel warmup finished in 0.00s.


Capturing CUDA graphs (FULL): 100%|██████████| 70/70 [00:08<00:00,  8.73it/s]


(EngineCore pid=22330) INFO 10-10 14:38:10 [model_runner.py:1087] Graph capturing finished in 24 secs, took 0.33 GiB
(EngineCore pid=22330) INFO 10-10 14:38:10 [gpu_worker.py:892] CUDA graph pool memory: 0.33 GiB (actual), 0.71 GiB (estimated), difference: 0.38 GiB (114.4%).
(EngineCore pid=22330) INFO 10-10 14:38:10 [gpu_worker.py:955] Free memory on device (14.46/14.56 GiB) on startup. Desired GPU memory utilization is (0.85, 12.38 GiB). Actual usage is 3.39 GiB for consumed memory (weights + non-torch), 1.2 GiB for peak activation, and 0.33 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=7844299469` (7.31 GiB) to fit into requested memory, or `--kv-cache-memory=10079892992` (9.39 GiB) to fully utilize gpu memory. Current kv cache memory in use is 7.78 GiB.
(EngineCore pid=22330) INFO 10-10 14:38:11 [jit_monitor.py:84] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=22330) INFO 10-10

In [11]:
# 1. Register the LoRA adapters with unique integer IDs and request handles
lora_requests = [
    LoRARequest("adapter_r4", 1, adapters["r4"]),
    LoRARequest("adapter_r8", 2, adapters["r8"]),
    LoRARequest("adapter_r16", 3, adapters["r16"])
]

# 2. Define test prompt (following your formal academic text conversion task)
test_prompt = "<|im_start|>user\nConvert the following casual message into a formal academic sentence.\nInput: PagedAttention stops GPU memory from being wasted when generating text.<|im_end|>\n<|im_start|>assistant\n"

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=64
)

print("=== Executing Multi-Adapter vLLM Generation ===")
print(f"Input Prompt: {test_prompt}\n")

# 3. Generate outputs dynamically across the base model and each rank adapter
# Base model request (no lora_request)
base_output = llm.generate([test_prompt], sampling_params=sampling_params)
print(f"[Base Model Output]:\n{base_output[0].outputs[0].text.strip()}\n")

# Multi-LoRA requests routing through the same base engine instance
for req in lora_requests:
    outputs = llm.generate([test_prompt], sampling_params=sampling_params, lora_request=req)
    generated_text = outputs[0].outputs[0].text.strip()
    print(f"[{req.lora_name} Output]:\n{generated_text}\n")

=== Executing Multi-Adapter vLLM Generation ===
Input Prompt: <|im_start|>user
Convert the following casual message into a formal academic sentence.
Input: PagedAttention stops GPU memory from being wasted when generating text.<|im_end|>
<|im_start|>assistant




Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.34s/it, est. speed input: 25.36 toks/s, output: 15.66 toks/s]

[Base Model Output]:
The software PagedAttention effectively prevents GPU memory from being inefficiently utilized during the generation of text.



Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.39s/it, est. speed input: 14.23 toks/s, output: 10.46 toks/s]

[adapter_r8 Output]:
PagedAttention effectively prevents GPU memory wastage by ensuring that only necessary model parameters are loaded into memory during text generation.



Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.63s/it, est. speed input: 12.95 toks/s, output: 9.14 toks/s]

[adapter_r16 Output]:
PagedAttention effectively prevents GPU memory wastage during text generation by ensuring that only necessary data is retained in memory.



In [14]:
import json

# Assuming you ran your multi-adapter inference in a loop like this:
results_data = {
    "prompt": test_prompt,
    "base_model": base_output[0].outputs[0].text.strip(),
    "adapters": {}
}

# Run inference and store results dynamically
for req in lora_requests:
    outputs = llm.generate([test_prompt], sampling_params=sampling_params, lora_request=req)
    results_data["adapters"][req.lora_name] = outputs[0].outputs[0].text.strip()

# Save to file
with open("vllm_evaluation_results.json", "w") as f:
    json.dump(results_data, f, indent=4)

print("Results saved successfully as 'vllm_evaluation_results.json'!")

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.02s/it, est. speed input: 33.62 toks/s, output: 24.72 toks/s]


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.04it/s, est. speed input: 35.66 toks/s, output: 26.22 toks/s]


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.10it/s, est. speed input: 37.48 toks/s, output: 26.45 toks/s]

Results saved successfully as 'vllm_evaluation_results.json'!
